# chrisMain — `EyeMovementTrajectoryAlternatingBackground`

Workflow notebook **tuned for the `EyeMovementTrajectoryAlternatingBackground`
protocol** (short name: `eye_movement_alt_bg`). All defaults — QC
thresholds, condition keys, movie-repeat cycle length, population
comparison axis — assume this protocol. **For a different protocol,
copy this notebook and adapt §4 thresholds + §11 analyzer choices;
do not edit chrisMain in place.**

## Sections at a glance

- **Setup (§1–§3)** — list protocol datafiles, pick one, build the
  `(StimBlock, ResponseBlock, AnalysisChunk)` pipeline.
- **QC + archive (§4 → §5 → §6/§9)** — automated protocol QC →
  optional click-through visual QC → per-cell PNG archive (single
  date or batch over many dates).
- **Spike-sorting QC (§7, §8)** — confirm spikes are assigned to
  the right cell. §7 writes static PNGs (good for batch / remote
  review); §8 is the interactive ipywidgets GUI (windowed loading,
  bandwidth meter — designed for remote NAS sessions).
- **Offline store (§10)** — pack QC-good cells into a single HDF5
  per date so subsequent sessions skip DataJoint + the SSD pipeline.
- **Analyses (§11–§12)** — protocol-specific offline analyses
  (`retinanalysis.protocols.eye_movement_alt_bg`) per date, then
  cross-date pooling.

## Workflow order (first time through)

1. Run §1 → §3 to build the pipeline for a single date.
2. Run §4 to compute `qc.csv` (auto firing-rate + silent-epoch gates).
3. Run §6 (or §9 for batch) to render the per-cell PNG archive.
4. Run §5 to tag cells `good` / `bad` interactively → `visual_qc.csv`.
5. Re-run §6 / §9: the archive now prunes to the curated set.
6. (Optional) Run §7 or §8 to sanity-check the sort itself.
7. Run §10 to write `offline.h5`, then §11 / §12 for analyses.

Sections marked **(optional)** can be skipped on a first pass.

## Reference

- Repo conventions: `CLAUDE.md` at the repo root.
- Removed exploratory cells (single-cell STA/EI inspection, regen
  stimulus + canvas overlay, raster/PSTH spot checks, manual rig
  calibration, EI-match diagnostics) are recoverable from git history:
  `git show <pre-consolidation-sha>:demos/chrisMain.ipynb`.

In [1]:
import retinanalysis as ra
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import os

## 1. Find all experiments that ran `EyeMovementTrajectoryAlternatingBackground`

`ra.get_datasets_from_protocol_names()` does a **lowercase substring
match** against the protocol registry, so the short query
`'AlternatingBackground'` catches the full Java class name
(`edu.washington.riekelab.turner.protocols.EyeMovementTrajectoryAlternatingBackground`)
without you typing it. The result has **one row per (exp_name,
datafile_name)** so multiple datafiles of the same protocol on one date
are already separated.

We then filter to experiments whose Kilosort output is actually on
disk under `ra.ANALYSIS_DIR` — registry rows without local sort data
are dropped silently.

In [2]:
# Protocol-registry rows for this protocol, filtered to dates with
# Kilosort output on disk. ra.find_available_datasets handles both steps
# (DJ query + os.listdir intersect) so §1 and §9 stay in sync.
exp_search = ra.find_available_datasets('AlternatingBackground')

print(f'{len(exp_search)} usable datafile(s) found across '
      f'{exp_search.exp_name.nunique()} experiment(s).')
display(exp_search)



Found 1 protocols matching "alternatingbackground":
['edu.washington.riekelab.turner.protocols.EyeMovementTrajectoryAlternatingBackground']

Found 22 experiments, 31 epoch blocks.

26 usable datafile(s) found across 17 experiment(s).


,exp_name,datafile_name,NDF,chunk_name,protocol_name,is_mea,data_dir,group_label,experiment_id,protocol_id,group_id,block_id,chunk_id
0,20220823C,data035,NaN,chunk5,edu.washington.riekelab.turner.protocols.EyeMo...,1,20220823C/data035,Eye movement trajectory alt background rod,22,41,315,866,45
1,20221101C,data018,0.0,chunk2,edu.washington.riekelab.turner.protocols.EyeMo...,1,20221101C/data018,Eye movement alt background,27,41,440,1047,56
2,20221101C,data025,0.0,chunk3,edu.washington.riekelab.turner.protocols.EyeMo...,1,20221101C/data025,Eye movement alt background,27,41,442,1054,57
3,20221101C,data032,0.0,chunk3,edu.washington.riekelab.turner.protocols.EyeMo...,1,20221101C/data032,Eye movement alt background,27,41,444,1061,57
4,20221123C,data027,2.0,chunk4,edu.washington.riekelab.turner.protocols.EyeMo...,1,20221123C/data027,CC eye movement alt background ndf2.0,30,41,530,1179,69
5,20230214C,data018,3.0,chunk2,edu.washington.riekelab.turner.protocols.EyeMo...,1,20230214C/data018,eye movement alt background ndf 3.0,39,41,781,1484,97
6,20230313C,data017,3.0,chunk2,edu.washington.riekelab.turner.protocols.EyeMo...,1,20230313C/data017,eye movement alt background ndf 3.0,42,41,878,1609,108
7,20230313C,data019,3.0,chunk2,edu.washington.riekelab.turner.protocols.EyeMo...,1,20230313C/data019,Var Mean Drift Grating ndf 3.0,42,41,879,1611,108
8,20230502C,data018,2.0,chunk2,edu.washington.riekelab.turner.protocols.EyeMo...,1,20230502C/data018,ndf 2.0 eye movement alt background,52,41,1077,1855,156
9,20230725C,data036,2.0,chunk4,edu.washington.riekelab.turner.protocols.EyeMo...,1,20230725C/data036,ndf 2 eye movement alt background,65,41,1397,2264,205


## 2. Pick a (date, datafile)

Edit `ENTRY_INDEX` below to a row index from §1's table. The notebook
binds `exp_name` and `datafile_name` together from that one row, so
they can't drift apart. To switch dates, just change the index and
re-run from §3 down.

The cell also displays a short summary of all datafiles recorded on
the chosen date (capped at 10 rows) for context — useful when checking
that you picked the right run of `EyeMovementTrajectoryAlternatingBackground`
rather than an adjacent protocol.

In [ ]:
ENTRY_INDEX = 19 # <-- EDIT ME: row index in exp_search above

# ---- Resolve the picked (exp_name, datafile_name, protocol_name) ------
_entry = exp_search.iloc[ENTRY_INDEX]
exp_name        = _entry['exp_name']
datafile_name   = _entry['datafile_name']
protocol_name   = _entry['protocol_name']
print(f'Selected entry {ENTRY_INDEX}:')
print(f'  exp_name      = {exp_name}')
print(f'  datafile_name = {datafile_name}')
print(f'  protocol_name = {protocol_name}')
print(f'  group_label   = {_entry.get("group_label", "<none>")}')

# ---- Context: every run of THIS protocol on the picked date ------------
# Some days have multiple datafiles of EyeMovementTrajectoryAlternatingBackground
# (different NDFs, repeats, turner vs other lab variants). Show just
# those rows so it is unambiguous which one ENTRY_INDEX selected --
# the picked row is marked with a "*".
_proto_on_day = exp_search[exp_search['exp_name'] == exp_name].copy()
_proto_on_day.insert(
    0, 'picked',
    ['*' if i == ENTRY_INDEX else '' for i in _proto_on_day.index],
)
print(f'\n{len(_proto_on_day)} AlternatingBackground datafile(s) on '
      f'{exp_name} (picked row marked *):')
_proto_cols = [c for c in ['picked', 'exp_name', 'datafile_name',
                            'protocol_name', 'NDF', 'chunk_name',
                            'group_label']
                if c in _proto_on_day.columns]
display(_proto_on_day[_proto_cols])

# ---- Optional: full day summary (every protocol), for orientation ------
# Capped so long experiments do not blow up the notebook output.
# Increase the cap if you need to inspect more rows (or call
# ra.get_exp_summary(exp_name) directly to see all of them).
SUMMARY_HEAD_N = 10
_sum_df = ra.get_exp_summary(exp_name)
# Show columns that disambiguate same-day rows; keep this view narrow.
_cols = [c for c in ['data_dir', 'protocol_name', 'chunk_name',
                     'group_label', 'start_time']
         if c in _sum_df.columns]
print(f'\nfull experiment summary: {len(_sum_df)} datafiles total '
      f'(showing first {min(SUMMARY_HEAD_N, len(_sum_df))})')
display(_sum_df[_cols].head(SUMMARY_HEAD_N))

# ---- Output-folder convention for downstream cells ---------------------
# All sections that write to disk (QC, visual QC, single archive,
# sorting-QC, offline store) live under
#   <OUTPUT_DIR>/<exp_name>/<protocol_subdir>/
# The default protocol_subdir is the protocol short name + the datafile
# name (e.g. eye_movement_alt_bg_data018), so two protocol runs on the
# same date cannot collide. Override protocol_subdir to a literal string
# if you want a custom folder name; set append_datafile_to_subdir = False
# to use the bare protocol name.
protocol_subdir = None              # None -> auto (see append_datafile_to_subdir)
append_datafile_to_subdir = True    # appends _<datafile> to the protocol short name


## 3. Mirror Vision files, then build (or load) the pipeline

Building the pipeline pulls Vision files (`.ei`, `.neurons`, `.params`,
`.classification.txt`, …) for one protocol datafile and one noise
chunk — ~1 GB total. It also runs DataJoint queries and the
`cluster_match` EI alignment. When `find_path` resolves to a remote
NAS, every kernel restart pays both costs again.

Two-tier cache covers both:

### Step 3a — local file mirror (one-time bandwidth fix)

`ra.mirror_to_local_cache(exp_name, datafile_name, chunk_name)` copies
the Vision files into `~/.cache/retinanalysis/` (override the path via
the `RA_LOCAL_CACHE_ROOT` environment variable). The `local_cache`
tier sits at the top of `find_path`'s priority list, so any
subsequent Vision read transparently uses the local copy. The huge
`.sta` raw STA movie is skipped by default — pass `include_sta=True`
to bring it.

### Step 3b — pipeline pkl cache (skip rebuild on restart)

`ra.create_mea_pipeline_cached(exp_name, datafile_name, ...)` is a
drop-in for `ra.create_mea_pipeline`. On first call: builds the
pipeline as usual and pickles it to
`<LOCAL_CACHE_ROOT>/pipelines/<key-hash>.pkl`. Subsequent calls with
the same EI-match knobs return the cached object directly — no
DataJoint queries, no `cluster_match` recompute. The pkl reload still
materializes per-block `vcd` objects, but with §3a's mirror those
reads are local.

**Cache key includes all EI-match knobs** (`ei_corr_cutoff`,
`ei_match_method`, `ei_use_isi`, `ei_use_timecourse`,
`ei_n_removed_channels`). Change any of them → a fresh cache entry
gets created automatically. To force a rebuild with the same knobs,
flip `OVERWRITE_PIPELINE_CACHE = True` at the top of §3b, or just
delete the printed pkl path.

### What the built pipeline gives you

- **`stim_block`** — `MEAStimBlock`: stimulus parameters and frame
  timing for the protocol datafile (preTime/stimTime/tailTime,
  `currentImageName`, `currentBackgroundScale`, eye-movement
  trajectory).
- **`response_block`** — `MEAResponseBlock`: spike times pulled from
  the Kilosort output (one list per cell × epoch).
- **`analysis_chunk`** — noise chunk: STAs, RF params, EIs, ISIs,
  timecourses, classification.
- **EI cluster-match** — `match_dict` (noise → protocol) and
  `corr_dict` (per-pair correlation). After construction,
  `response_block.df_spike_times` carries `noise_id` and `cell_type`.

### Pitfall

Older experiments are often sorted with `kilosort2` (not `kilosort2.5`)
— §3b auto-detects available versions. If both exist, `kilosort2.5`
wins. macOS AppleDouble dotfiles (`._kilosort*.classification.txt`)
are explicitly skipped.

### Skip the cache when

- You're working off a local SSD already (mirror is harmless but
  redundant copy work).
- You change a kwarg that *isn't* in the cache key (e.g. `ls_params`)
  — pass `OVERWRITE_PIPELINE_CACHE = True`.

### Bandwidth gauge (session-wide)

Reads ALWAYS prefer local tiers over the NAS (auto-detected from
mount fstype — SSDs over `smbfs`/`nfs`/`afpfs`), so files that
exist on both ChrisProSSD and the NAS are read from the SSD. Files
that only exist on the NAS fall through to the network tier; every
such resolution charges `ra.network_bytes_resolved()` so you can
see a running tally.

Drop this in any cell to see the live gauge:

```python
from IPython.display import display
display(ra.network_bandwidth_gauge_widget())
```

or just print:

```python
ra.print_network_gauge()
```

Cache hits at the local_cache tier (after §3a's mirror) never
charge the counter, so a fully-mirrored experiment goes through
the whole pipeline with the gauge staying at zero. The gauge is
an *upper bound* — repeat lookups credit the same bytes again
even if the OS page cache served them for free.

In [93]:
# Step 3a — mirror this date's Vision files to the local cache.
# Safe to re-run: files already present are skipped (size + mtime match).
# Skip / comment out this cell when working off a local SSD already.
import os

# Auto-detect ss_version + noise chunk for the mirror so the user
# doesn't have to set them twice (§3b will resolve them again the
# same way). For the noise chunk we use the same MEAStimBlock auto-pick
# as the pipeline builder below — the auto-pick is cheap (a small DJ
# query) and doesn't touch NAS Vision files.
# Pick kilosort version by project priority: kilosort2.5 > kilosort2 >
# kilosort4 (then any other). Centralized in ra.detect_ss_version so the
# notebook and library agree.
_ss_version = ra.detect_ss_version(exp_name, datafile_name)

# Pin a noise chunk if you want; otherwise we resolve the auto-pick once.
from retinanalysis.classes.stim import MEAStimBlock
_tmp = MEAStimBlock(exp_name, datafile_name, verbose=False)
_chunk = _tmp.nearest_noise_chunk

print(f'Mirror plan:  {exp_name} / {datafile_name} (ss={_ss_version}) + chunk {_chunk}\n')

# First: a dry-run compare so the user sees what would be transferred.
# If everything is in sync, mirror_to_local_cache below will be a no-op.
ra.compare_cache_vs_source(
    exp_name, datafile_name=datafile_name, chunk_name=_chunk,
    ss_version=_ss_version, include_sta=False,
)
print()

with ra.bandwidth_scope('Mirror (read from upstream tier)'):
    mirror_report = ra.mirror_to_local_cache(
        exp_name,
        datafile_name=datafile_name,
        chunk_name=_chunk,
        ss_version=_ss_version,
        include_sta=False,    # raw .sta is ~9 GB and not needed by the pipeline
        verbose=True,
    )
print(f'\nLocal cache root: {ra.LOCAL_CACHE_ROOT}')
print(f'  copied this run: {mirror_report["bytes_copied_total"] / 1e6:.1f} MB')
print(f'  total on disk  : {mirror_report["bytes_total"] / 1e6:.1f} MB')


d_display will keep the first one: 59.0
Mirror plan:  20250306C / data030 (ss=kilosort2.5) + chunk chunk4

Comparing cache vs canonical source for 20250306C
  [data/data030]
    src   = /Volumes/ChrisProSSD/data/sorted/20250306C/data030/kilosort2.5
    cache = /Users/chrischen/.cache/retinanalysis/data/20250306C/data030/kilosort2.5
    in-sync          : 0 files
    cache-only       : 4 files (source no longer has them; nothing to copy):
      ? data030.ei
      ? data030.globals
      ? data030.neurons
      ? data030.noise
  [analysis/chunk4]
    src   = /Volumes/ChrisProSSD/analysis/20250306C/chunk4/kilosort2.5
    cache = /Users/chrischen/.cache/retinanalysis/analysis/20250306C/chunk4/kilosort2.5
    in-sync          : 8 files

  → cache is fully in sync; no transfer needed.

Mirroring Vision files for 20250306C into /Users/chrischen/.cache/retinanalysis
  [data/data030] source missing: /Volumes/ChrisProSSD/data/sorted/20250306C/data030/kilosort2.5  — skip
  [analysis/chunk4] cache

In [94]:
# Step 3b — build the pipeline (or load from local pkl cache).
#
# create_mea_pipeline_cached() is a drop-in for ra.create_mea_pipeline()
# that pickles the built pipeline under <LOCAL_CACHE_ROOT>/pipelines/
# keyed by every build kwarg that affects the output. Subsequent kernel
# restarts with the same kwargs reload from the pkl — no DataJoint
# queries, no cluster_match recompute. With §3a's mirror in place,
# the per-block VCD reload that the pkl path triggers is also local.
#
# Cache invalidation is automatic: change any EI-match knob below
# (ei_corr_cutoff, ei_match_method, ei_use_isi, ei_use_timecourse,
# ei_n_removed_channels) and the next call gets a fresh build. Pass
# overwrite=True to force a rebuild with the same kwargs.

# Centralized in ra.detect_ss_version so the notebook and library agree
# (analyze_experiment uses the same helper).
ss_version = ra.detect_ss_version(exp_name, datafile_name)

# ---- USER INPUT --------------------------------------------------------
# Noise chunk to use for cell typing + EI matching.
#   None  → MEAStimBlock.nearest_noise_chunk (closest in time, either direction)
#   'chunkN' → pin explicitly. Bypasses get_nearest_noise.
noise_chunk_name = None
typing_file_name = None           # e.g. 'kilosort2.5.classification.txt', or None for first match

# EI cluster-match knobs (forwarded to vision_utils.cluster_match).
ei_corr_cutoff       = 0.6       # minimum max-correlation to accept a match (0.6 looser, 0.9 stricter)
ei_match_method      = 'all'      # 'all' (max of three), 'full', 'space', 'power'
ei_use_isi           = False      # also require ISI corr ≥ 0.3
ei_use_timecourse    = False      # also require RGB timecourse corr ≥ 0.3
ei_n_removed_channels = 1         # drop this many top-amplitude electrodes per EI before correlating

OVERWRITE_PIPELINE_CACHE = False   # set True to force a rebuild even on a cache hit
# ------------------------------------------------------------------------

# Resolve noise chunk: explicit override > auto-pick. Also sanity-checks
# against the DB record written at ingest time so a mismatch is flagged.
noise_chunk_name, db_chunk_name, _chunk_warning = ra.resolve_noise_chunk(
    exp_name, datafile_name, override=noise_chunk_name,
)
if db_chunk_name is not None:
    print(f'database chunk_name for {datafile_name}: {db_chunk_name}')
print(f'using noise chunk: {noise_chunk_name}')
if _chunk_warning:
    print(f'\n*** ALERT: {_chunk_warning} ***\n')

# Resolve typing file (strict=True → raises FileNotFoundError on miss;
# walks the local-cache → SSD → NAS tiers; skips macOS AppleDouble dotfiles).
typing_file_name = ra.pick_typing_file(
    exp_name, noise_chunk_name, ss_version, preferred=typing_file_name,
)

print(f'ss_version: {ss_version}')
print(f'typing file: {typing_file_name}')
print(f'EI match: method={ei_match_method!r} cutoff={ei_corr_cutoff} '
      f'use_isi={ei_use_isi} use_timecourse={ei_use_timecourse} '
      f'n_removed_channels={ei_n_removed_channels}')

# Show where the pkl will go, so the user knows what to delete if they
# want to force a rebuild without flipping OVERWRITE_PIPELINE_CACHE.
_pkl_path = ra.pipeline_cache_path(
    exp_name, datafile_name,
    ss_version=ss_version, analysis_chunk_name=noise_chunk_name,
    typing_file=typing_file_name,
    ei_corr_cutoff=ei_corr_cutoff, ei_match_method=ei_match_method,
    ei_use_isi=ei_use_isi, ei_use_timecourse=ei_use_timecourse,
    ei_n_removed_channels=ei_n_removed_channels,
)
print(f'pkl cache: {_pkl_path}')

with ra.bandwidth_scope('Pipeline build', also_print_total=True):
    pipeline = ra.create_mea_pipeline_cached(
        exp_name,
        datafile_name,
        overwrite=OVERWRITE_PIPELINE_CACHE,
        verbose=True,
        ss_version=ss_version,
        typing_file=typing_file_name,
        analysis_chunk_name=noise_chunk_name,
        ei_corr_cutoff=ei_corr_cutoff,
        ei_match_method=ei_match_method,
        ei_use_isi=ei_use_isi,
        ei_use_timecourse=ei_use_timecourse,
        ei_n_removed_channels=ei_n_removed_channels,
    )

stim_block      = pipeline.stim
response_block  = pipeline.resp
analysis_chunk  = pipeline.analysis_chunk

print(f'\nNoise chunk used: {analysis_chunk.chunk_name}')
print(f'Cells in noise chunk: {len(analysis_chunk.cell_ids)}')
print(f'Cells in protocol datafile: {len(response_block.cell_ids)}')
print(f'Cells matched by EI: {len(pipeline.match_dict)}')
print(f'ei_match_config: {pipeline.ei_match_config}')


d_display will keep the first one: 59.0
database chunk_name for data030: eye_move
using noise chunk: chunk4

*** ALERT: DB chunk 'eye_move' differs from chunk being used ('chunk4'). If the DB record is correct, pin noise_chunk_name='eye_move'. ***

ss_version: kilosort2.5
typing file: kilosort2.5.classification.txt
EI match: method='all' cutoff=0.6 use_isi=False use_timecourse=False n_removed_channels=1
pkl cache: /Users/chrischen/.cache/retinanalysis/pipelines/20250306C__data030__kilosort2.5__de1567d8c7.pkl
Building pipeline (no cache hit at 20250306C__data030__kilosort2.5__de1567d8c7.pkl) …
Initializing StimBlock for 20250306C block 3229
For Rig C 20250306C:
d_display will keep the first one: 59.0
{'disp_type': 'LCR', 'mu_per_pixel': 3.34, 'n_ht': 1140, 'n_wt': 1824, 'mean_frame_rate': 59.94154881781792, 'stage_frame_rate': np.float64(59.0), 'mea_rotation_deg': 90.0}
Using user-specified noise chunk for data030: chunk4.

Initializing ResponseBlock for 20250306C block 3229
If this is 

## 3c. Merge duplicate clusters before QC

Kilosort sometimes splits one physical cell across two clusters with
near-identical EIs. Counting them as separate cells inflates QC pass
rates, biases mosaics, and double-weights downstream analyses. We
merge them once, right here, so every later section (§4 QC, §5
visual QC, §6/§9 archive, §7/§8 sorting QC, §10 offline store) sees
each biological cell exactly once.

**What `ra.dedup_pipeline` does:** computes pairwise EI correlations on
the protocol side, builds connected-components of clusters with
`corr ≥ ei_threshold`, picks a representative per group (highest EI
amplitude by default), and **unions** the others' spike trains into
it (with 0.5 ms refractory dedup to absorb double-counted spikes).
Untyped clusters and pairs across different cell types are skipped.

**Mutates `pipeline.resp` in place.** Idempotent: rerunning after a
merge finds zero new groups.

**Knobs you'd actually change:**

| knob | default | when to change |
|---|---|---|
| `ei_threshold` | 0.85 | lower (0.75) catches more splits at the cost of more false merges; raise (0.92) to be strict |
| `merge_strategy` | `'union'` | `'drop'` to keep only the rep's train when you suspect the duplicate is actually a different cell |
| `refractory_ms` | 0.5 | match your sorter's refractory window; 0 to disable refractory dedup |
| `skip_untyped` | True | False to merge untyped clusters too (rarely useful) |

Same defaults as `analyze_experiment` (§6/§9), so the namespace
pipeline and the archived PNGs stay consistent.

In [44]:
# §3c — Dedup the pipeline IN PLACE before any QC/analysis.
# `analyze_experiment` (§6/§9) and `load_or_build_offline` (§10) run
# the same step internally — this cell exists so the in-notebook
# `pipeline` / `response_block` objects also reflect the merge, so
# §4 QC, §7/§8 sorting QC, and any ad-hoc inspection you do here
# agree with the archive on disk.
DEDUP_EI_THRESHOLD   = 0.85
DEDUP_MERGE_STRATEGY = 'union'   # 'union' merges spike trains; 'drop' keeps rep only
DEDUP_REFRACTORY_MS  = 0.5
DEDUP_SKIP_UNTYPED   = True

_n_cells_before = len(response_block.df_spike_times)
dedup_log = ra.dedup_pipeline(
    pipeline,
    ei_threshold=DEDUP_EI_THRESHOLD,
    merge_strategy=DEDUP_MERGE_STRATEGY,
    refractory_ms=DEDUP_REFRACTORY_MS,
    skip_untyped=DEDUP_SKIP_UNTYPED,
    verbose=True,
)
_n_cells_after = len(response_block.df_spike_times)
print(f'\nProtocol cells: {_n_cells_before} -> {_n_cells_after} '
      f'({_n_cells_before - _n_cells_after} merged into representatives)')

# Per-group audit (rep cell, who got merged, spike counts before/after).
# Empty when nothing was merged.
if not dedup_log['protocol'].empty:
    display(dedup_log['protocol'])

Deduplicating protocol side (protocol)…
find_duplicate_groups: 2 group(s), 4 cells affected, 2 would be dropped as duplicates (EI≥0.85, same-type)
apply_dedup: kept 2 representative(s), dropped 2 duplicate cell(s); recovered 935 spikes via union (0.5 ms refractory)

Protocol cells: 867 -> 865 (2 merged into representatives)


,group,representative,dropped,representative_amp,cell_type,n_spikes_rep_before,n_spikes_dropped_total,n_spikes_rep_after,n_spikes_added_to_rep
0,"(423, 494)",423,"(494,)",288.651886,OffP,11512,742,11622,110
1,"(514, 1165)",1165,"(514,)",174.092834,A1,4098,826,4923,825


## 4. Per-cell QC inside the protocol (`EyeMovementTrajectoryAlternatingBackground`)

A cell can pass classification on the noise chunk and still misbehave
inside a long protocol like this one — drift off, drop out for runs of
trials, or barely fire. `protocol_qc.block_qc_metrics()` returns a
per-cell metrics DataFrame; `filter_cells_by_qc()` adds a boolean
`passes` column.

**Output**: `<OUTPUT_DIR>/<exp>/<protocol_subdir>/qc.csv` — the **initial
good/bad tagging** for every cell. §5 (visual QC), §6/§9 (archives)
and §10 (offline store) all honor it.

### Two automated gates do most of the work

- **Adaptive firing rate.** A cell passes when ≥ `min_frac_epochs_above_rate`
  (default **80%**) of its epochs hit the rate threshold
  (`min_rate_hz × epoch_duration_s`, default 1 Hz). Scales with epoch
  length so the same defaults work across 5 s, 30 s, and 60 s
  protocols (typical for this protocol is ~60 s).
- **Silent-epoch survival.** A cell passes when ≥
  `min_frac_non_silent_epochs` (default **2/3**) of its epochs have
  ≥1 spike — "drop the silent epochs and keep the cell if at least
  two-thirds of its trials survive" without actually dropping epochs.

### Tuning

`OVERWRITE_QC = False` by default: a prior `qc.csv` is loaded as-is.
Flip to `True` after editing `MIN_RATE_HZ` / `MIN_FRAC_EPOCHS` /
`MIN_FRAC_NON_SILENT` to recompute and overwrite. The summary printed
at the bottom reports pass rate per cell type and the first few failing
cells with their gate scores — quick way to see whether the gates are
biased against a specific type.

### Other reportable metrics (off by default; gate by setting thresholds)

`min_count_per_epoch`, `cv_count`, `fano`, `silent_trial_frac`,
`silent_run_max`, `drift_score`, `reliability_r` (split-half PSTH; off
by default for mixed-condition protocols like this one).

In [ ]:
# §4 — Protocol QC. ra.load_or_compute_protocol_qc reads qc.csv if it
# exists (overwrite=False), otherwise runs block_qc_metrics +
# filter_cells_by_qc and writes it. Same protocol_subdir resolution
# as §6/§9, so paths stay in sync.

# ---- USER INPUT --------------------------------------------------------
OVERWRITE_QC = False               # True → recompute even when qc.csv exists
MIN_RATE_HZ = 1                    # firing-rate floor in spikes/s
MIN_FRAC_EPOCHS = 0.8              # fraction of epochs that must meet that rate
MIN_FRAC_NON_SILENT = 2.0 / 3.0    # cell kept iff ≥ this fraction of epochs has ≥1 spike
# ------------------------------------------------------------------------

qc = ra.load_or_compute_protocol_qc(
    response_block, exp_name,
    protocol_subdir=protocol_subdir,
    append_datafile_to_subdir=append_datafile_to_subdir,
    datafile_name=datafile_name,
    overwrite=OVERWRITE_QC,
    min_rate_hz=MIN_RATE_HZ,
    min_frac_epochs=MIN_FRAC_EPOCHS,
    min_frac_non_silent=MIN_FRAC_NON_SILENT,
    verbose=True,
)

# Show the most informative failures.
fails = qc[~qc.passes].sort_values('frac_non_silent_epochs')
print(f'\nFirst few failing cells ({len(fails)} total):')
display(fails[['cell_id', 'cell_type', 'n_epochs', 'mean_rate_hz',
               'frac_epochs_above_rate', 'frac_non_silent_epochs',
               'silent_run_max', 'drift_score']].head().round(2))


overwrite=True: recomputing and overwriting /Volumes/ChrisProSSD/retinanalysis_output/20250306C/eye_movement_alt_bg_data030/qc.csv


## 5. Visual QC (optional) — click through each cell, tag good/bad

**This step is optional.** §4 wrote an automated QC pass/fail to
`qc.csv`. Use this section when you want to **further restrict** the
archive by eyeballing each cell's raster + PSTH.

### Iterative workflow

1. First time through, **skip this section** (no PNGs exist yet) and
   run §6 / §9 to build the initial archive.
2. Come back here once PNGs are on disk — `ra.browse_cells_qc(exp_name)`
   opens an ipywidgets panel that pages through cells (raster left,
   PSTH right) with `Good` / `Bad` / `Prev` / `Next` buttons. Each
   click upserts a row in
   `<OUTPUT_DIR>/<exp>/<protocol_subdir>/visual_qc.csv` — the session
   is resumable.
3. Re-run §6 / §9. They auto-detect `visual_qc.csv` and restrict the
   per-cell PNG render to cells tagged `good`. `cell_match.csv` is
   left comprehensive so downstream EI joins still see the full
   population.

### Downstream selection (no change needed)

```python
cells = ra.select_good_cells()   # uses visual_qc.csv if present, else QC-pass set
```

### Invariant

`visual_qc.csv` is **written only by this GUI**. `analyze_experiment`,
`save_per_cell_plots`, `save_cell_match`, and `save_protocol_qc` are
all read-only with respect to it (audited in
`tests/test_visual_qc_invariant.py`).

Requirements: `ipywidgets` (already in the `retinanalysis` kernel).

In [91]:
# Launch the per-cell GUI for the date picked in §2. If no PNGs exist
# yet, the widget prints a message and returns — run §6 (single date) or
# §9 (batch) first to build the archive, then come back here to tag.
#
# `datafile_name` + `protocol_subdir` from §2 select the same archive
# subdir §6 / §9 wrote into (e.g. eye_movement_alt_bg_data018/).
ra.browse_cells_qc(
    exp_name,
    datafile_name=datafile_name,
    protocol_subdir=protocol_subdir,
)


## 6. Archive the picked date (single date)

Run `ra.analyze_experiment(exp_name, datafile_name)` to write the
**full per-cell PNG archive** for the date selected in §2. Output goes
to `<OUTPUT_DIR>/<exp>/<protocol_subdir>/`:

| file | what it is |
|---|---|
| `mosaic.png` | composite: STA mosaic + temporal-filter + ISI rows |
| `index.csv` | one row per archived cell (`cell_id`, `cell_type`, …) |
| `cell_match.csv` | EI-match diagnostics per cell — kept comprehensive |
| `cells/<celltype>/cell_<id>_raster.png` | per-cell raster, condition-colored |
| `cells/<celltype>/cell_<id>_psth.png` | per-cell PSTH, condition-colored |

### Visual-QC integration is automatic

If `visual_qc.csv` exists for this experiment (from §5), the call
below **restricts the per-cell PNG step to cells tagged `good`** — no
extra flags needed. Otherwise it falls back to every cell that passed
§4's automated QC.

### Re-archiving prunes stale PNGs

`prune_stale=True` (default): any `cells/.../cell_<id>_*.png` whose
`cell_id` is *not* in the kept set (QC-pass ∩ visual-QC `good`) is
deleted on re-run. So tagging cells `bad` in §5 and re-running this
cell removes their PNGs from disk. Non-canonical files (e.g. a
`README`) are not touched.

### Pitfall: same-protocol-twice-in-one-day

If a date has **two datafiles of the same protocol**, the default
`protocol_subdir` (the short name, e.g. `eye_movement_alt_bg`) is the
same for both runs — the second would overwrite the first. Set
`append_datafile_to_subdir=True` (or pass an explicit
`protocol_subdir`) to disambiguate.

### Conditions used for raster + PSTH coloring

Auto-detected from this protocol's registry entry; for
`EyeMovementTrajectoryAlternatingBackground` it is
`currentBackgroundScale` (the low/high background pairing).

In [90]:
# Full archive for the date picked in §2. analyze_experiment now reads
# visual_qc.csv on its own (respect_visual_qc=True by default) and
# restricts the per-cell PNG step to cells tagged 'good' when the file
# exists — no extra notebook glue needed. overwrite=True regenerates
# every targeted PNG.

# The protocol_subdir / append_datafile_to_subdir defaults are set in §2
# so the same convention applies to §4 QC and §7/§8 sorting QC as well.

result = ra.analyze_experiment(
    exp_name,
    datafile_name=datafile_name,
    overwrite=True,
    fit_calibration=False,
    n_jobs=-1,
    verbose=True,
    protocol_subdir=protocol_subdir,
    append_datafile_to_subdir=append_datafile_to_subdir,
)
print(f'\nDone: {result["exp_name"]} / {result["datafile_name"]} — '
      f'QC-pass pool: {result["n_cells_passed_qc"]}/{result["n_cells_total"]}')
print(f'  output_dir: {result["output_dir"]}')



=== analyze_experiment(20240523C) ===
  datafile=data011  ss_version=kilosort2.5
  typing_file=kilosort2.5.classification.txt
If this is for an LED stimulus, be sure to set b_LED=True!

Error occurred while getting actual onset/offset times: unsupported operand type(s) for *: 'float' and 'NoneType'
It could be that frame_times_ms do not have the correct number of frames due to some error in frame detection.
Check the frame monitor sample rate! On MEA Rigs, prefer 1k, errors likely with 10k.
Spatial maps were not loaded
  cell_types_used=['OnP', 'OffP', 'OnM', 'OffM']  (skipped [])
  calibration: none on disk; using geometric fallback
  QC: 429/690 cells pass (62%)
  protocol_subdir → 'eye_movement_alt_bg_data011'
  qc → /Volumes/ChrisProSSD/retinanalysis_output/20240523C/eye_movement_alt_bg_data011/qc.csv
  visual_qc → 183 tags (107 good, 76 bad); archive restricted to good set
  cell_match → /Volumes/ChrisProSSD/retinanalysis_output/20240523C/eye_movement_alt_bg_data011/cell_match.cs

## 7. Spike-sorting QC — static PNGs (batch / remote review)

PSTH + raster confirm that spike *times* are consistent with the
stimulus; they don't tell you whether **the spikes were assigned to
the right cell** in the first place. This cell samples a few cells
(QC-pass ∩ visual-QC `good`) per type and writes one **multi-row PNG
per cell**: each row is one full epoch (raster strip on top, 300-Hz
high-pass-filtered raw trace below, red dots at the cell's spike
times, snapped to the local trough in ±2 ms).

A clean sort: red dots land on visible spike waveforms in the trace.
A merge: extra waveforms in the trace with no red dot, *or* red dots
on flat baseline (template hits that aren't real spikes).

### Output

`<OUTPUT_DIR>/<exp>/<protocol_subdir>/sorting_qc_<protocol_short>_<datafile>/cell_proto<XXXX>_noise<YYYY>_<celltype>_sorting_qc.png`

The folder name stamps both the protocol short name and the datafile
so multiple runs of `EyeMovementTrajectoryAlternatingBackground` on
one date don't collide.

### When to prefer this over §8

- **Batch review of many cells / dates**: PNGs are reviewable offline
  and shareable.
- **Slow or metered connection**: a 4-epoch sample is heavy
  (~3.6 GB read from raw `.bin` per cell on a typical 60-s epoch). §8
  loads only a sub-window per click.

### When to prefer §8

- **Spot checks** of one cell at a time with interactive zoom.
- **Remote NAS sessions** — see the bandwidth chip in §8.

In [ ]:
# §7 — Sorting QC via raw traces, saved to disk as high-DPI PNGs.
# Samples from QC-pass ∩ visual-QC 'good' cells per type and writes one PNG
# per cell to <OUTPUT>/<exp>/<protocol_subdir>/sorting_qc_<protocol>_<datafile>/.
# Each PNG has N_EPOCHS full-width rows; every row = thin raster strip +
# 300 Hz HP-filtered trace with red marks at the cell's spike times.

CELL_TYPES         = ['OnP', 'OnM']    # which types to sample
N_CELLS_PER_TYPE   = 3                  # cells per type
N_EPOCHS           = 4                  # full epochs to show per cell
SAMPLE_STRATEGY    = 'random'           # 'random' (default) or 'top_rate'
RANDOM_SEED        = None               # int for reproducible sampling; None = fresh
DPI                = 250                # 200–300 is good for visual inspection
OVERWRITE_QC_PNGS  = True               # re-render existing PNGs

sample_df, png_paths = ra.sample_and_plot_sorting_qc(
    response_block,
    protocol_subdir=protocol_subdir if 'protocol_subdir' in dir() else None,
    append_datafile_to_subdir=(append_datafile_to_subdir
                                if 'append_datafile_to_subdir' in dir() else False),
    cell_types=CELL_TYPES,
    n_cells_per_type=N_CELLS_PER_TYPE,
    n_epochs=N_EPOCHS,
    sample_strategy=SAMPLE_STRATEGY,
    random_seed=RANDOM_SEED,
    dpi=DPI,
    overwrite=OVERWRITE_QC_PNGS,
)
print(f'\n→ wrote {len(png_paths)} PNG(s); open them with the system viewer.')


## 8. Interactive sorting-QC GUI (`ra.sorting_qc_gui`)

Notebook ipywidgets panel — same diagnostic as §7 but **one click =
one window**, on demand. Designed for remote-NAS sessions where
loading a full epoch (~1 GB at 20 kHz × 512-electrode 12-bit) is
wasteful.

### Controls (top to bottom)

| widget | what it does |
|---|---|
| **Cell** | dropdown over `QC-pass ∩ visual-QC good` for cell types `OnP`, `OnM` (extendable). Labels show cell type, protocol cell id, matched noise id, mean rate. |
| **Epoch** | which epoch of `EyeMovementTrajectoryAlternatingBackground` to load (~60 s each in this protocol). |
| **Electrode** | 1st / 2nd / 3rd top-amplitude electrode for the chosen cell's EI. Switching rank is **free** (`rt.data` already holds all 512 electrodes for the cached window). |
| **Window (s)** | slider + `start (s)` / `end (s)` text boxes (synced). Type for precise values; the estimated MB on the wire is shown next to the slider. |
| **Appearance** (accordion) | trace color / line width, spike color / marker size, HP cutoff (Hz), manual y-range (also acts as a y-axis scrollbar: drag the bar between handles to pan), figure width and height. **All re-render from cache with zero I/O.** |
| **View (X)** row | ⊕ zoom in · ⊖ zoom out · ◀ pan · pan ▶ — shrink/expand or shift the window by half its width. Zoom-in stays **inside the loaded data → free**; pan past the loaded window re-fetches. |
| **View (Y)** row | ⊕ zoom y · ⊖ zoom y · ▲ pan y · pan y ▼ · y auto-fit — adjust the voltage axis range. Pressing any of these auto-disables `y-axis auto`. **All free** — pure re-render. |
| **Load raw trace** | the main button that may trigger NAS bytes. Idempotent: re-clicking the same (epoch, window) is served from cache, and any sub-range of the loaded window also hits the cache. |
| **Overlay detected spikes** | toggle red dots + raster strip. Zero I/O. |
| **Bandwidth chip** | green = local mount (SSD, USB, …), amber = network mount (SMB / NFS / AFP). Ticks up only on real reads; reset button zeroes it. *Caveat*: macOS page-cache means this is an **upper bound** on wire bytes. |

### What re-fetches over the network vs what is free

| action | re-reads file? |
|---|---|
| First Load click for a new (cell, epoch, window) | **yes** |
| Same Load click again | no (idempotency guard) |
| Switch electrode rank | no |
| Toggle spike overlay | no |
| Any appearance widget (color, lw, HP cutoff, fig height, …) | no |
| **View (X)** ⊕ zoom in (or smaller window inside the loaded one) | no — served from cache |
| **View (X)** ⊖ zoom out / ◀ pan / pan ▶ past the loaded window | **yes** for the new bytes only |
| **View (Y)** any button (zoom / pan / reset) | no — pure re-render |

### Vector output

Plots are **always vector SVG** — browser pinch / Ctrl-+ stays crisp
at any zoom level. For a *native* pan / zoom-rectangle toolbar inside
the figure (with a draggable selection rectangle), `pip install
ipympl` then add `%matplotlib widget` at the top of the notebook;
the GUI surfaces a one-line hint when ipympl is absent.

In [14]:
from IPython.display import display

# Launches an ipywidgets panel for the pipeline built in §3.
# Pick cell → epoch → top-3 electrode → window (slider or
# FloatText) → 'Load raw trace'. Appearance accordion exposes
# trace/marker style and HP-cutoff frequency. The bandwidth
# meter at the bottom tracks cumulative MB read from disk.
#
# protocol_subdir / append_datafile_to_subdir come from §2 so the GUI
# reads qc.csv from the same folder §4/§6/§9 wrote it into
# (e.g. eye_movement_alt_bg_data030/qc.csv).
display(ra.sorting_qc_gui(
    response_block,
    protocol_subdir=protocol_subdir if 'protocol_subdir' in dir() else None,
    append_datafile_to_subdir=(append_datafile_to_subdir
                                if 'append_datafile_to_subdir' in dir() else False),
))

## 9. Run the archive for one or many dates (standalone)

**Self-contained section** — run cell 1 (imports), then jump here.
`ra.analyze_experiments(dates, protocol_search=...)` packages every
step the earlier cells did manually into a single call per date:
`ss_version` + typing file + datafile auto-detect, pipeline build,
optional rig calibration, type normalization, QC, composite
`mosaic.png` (with temporal-filter + ISI rows), and per-cell
`cell_<id>_raster.png` + `cell_<id>_psth.png`.

### Defaults assume `EyeMovementTrajectoryAlternatingBackground`

`PROTOCOL_SEARCH = "AlternatingBackground"` substring-matches the full
Java class name. Change it to run the same archive over a different
protocol family — but condition coloring and QC thresholds may need
tuning for that protocol.

### Visual-QC integration

For any date that already has a `visual_qc.csv` in its archive folder,
the batch driver restricts that date's per-cell PNG step to cells
tagged `good`. Pass `respect_visual_qc=False` to override.

### Save toggle

`SAVE_FIGURES = True` ⇒ render and **overwrite** all PNGs.
`SAVE_FIGURES = False` ⇒ list the batch only and stop. The user
prefers this explicit gate over an auto-detect "does the PNG exist?"
check — don't re-introduce implicit skipping.

### Failure handling

`on_error="log"` so one bad date is recorded in the returned summary
DataFrame instead of aborting the batch. `n_jobs=-1` uses every CPU
core for per-cell PNG rendering.

In [ ]:
# Section 9 is SELF-CONTAINED — you only need cell 1 (imports) to run it.
# It will query the protocol registry on its own and dispatch the archive
# pipeline over every experiment found, in parallel.

# ---- USER INPUT --------------------------------------------------------
# Yes/no: should we (re)save figures for every cell?
#   True  → render and OVERWRITE all PNGs (use this to refresh stale plots).
#           Per-date visual_qc.csv (if present) restricts the per-cell
#           PNG step to cells tagged 'good'.
#   False → skip the archive step entirely (just list the batch and stop).
SAVE_FIGURES = True
# ------------------------------------------------------------------------

PROTOCOL_SEARCH = 'AlternatingBackground'   # substring matched against protocol names

# Build the date list from the protocol registry, keeping only experiments
# whose sort output is present on disk.
_exp_search = ra.find_available_datasets(PROTOCOL_SEARCH)
batch_dates = _exp_search['exp_name'].unique().tolist()

# Subset variants — uncomment / adapt as needed:
# batch_dates = ['20220823C', '20221123C', '20230502C']                                            # hand-pick
# batch_dates = _exp_search.query("exp_name >= '20230101C'")['exp_name'].unique().tolist()        # by date
# batch_dates = _exp_search.query('NDF == 2.0')['exp_name'].unique().tolist()                     # by NDF

print(f'Batch run over {len(batch_dates)} dates: {batch_dates}')
print(f'SAVE_FIGURES = {SAVE_FIGURES}  '
      f'({"overwrite all PNGs" if SAVE_FIGURES else "skip archive step"})')

if not SAVE_FIGURES:
    print('SAVE_FIGURES is False — not calling ra.analyze_experiments. '
          'Set SAVE_FIGURES = True above to (re)render PNGs.')
else:
    results = ra.analyze_experiments(
        batch_dates,
        protocol_search=PROTOCOL_SEARCH,    # resolves datafile per date
        fit_calibration=False,              # True on first pass to seed calibrations
        overwrite=True,                     # resave every PNG (driven by SAVE_FIGURES)
        n_jobs=-1,                          # all CPU cores for per-cell rendering
        on_error='log',                     # keep going past per-date failures
        respect_visual_qc=True,             # restrict to good-tagged cells when present
        verbose=True,
    )
    display(ra.summarize_batch_results(results))


## 10. Offline data store (`offline.h5`) — build once, reload fast

After §5/§6 leaves a curated visual-QC set, `ra.load_or_build_offline`
packages everything an analysis needs — metadata, condition table,
per-cell spike times, smoothed PSTHs, STA fit, EI summary — into a
single HDF5 at
`<OUTPUT_DIR>/<exp>/eye_movement_alt_bg/offline.h5`. Subsequent
sessions just **load** the file; no DataJoint, no SSD pipeline rebuild.

- **First call**: builds the pipeline → runs §4 QC → intersects with
  `visual_qc.csv` (good cells only) → writes `offline.h5`. ~1–2 min/date.
- **Re-runs**: `ra.load_offline_data(exp)` returns an `OfflineDataset`
  in <1 s. Pass `overwrite=True` to rebuild from source.
- **Cross-date**: `ra.load_offline_many()` → `{exp_name: OfflineDataset}`
  for every date that has `offline.h5`.

### `OfflineDataset` API

| attribute / method | what it is |
|---|---|
| `ds.meta`, `ds.timing` | scalars (exp id, datafile, ndf, preTime/stimTime/sample_rate) |
| `ds.epochs` | DataFrame, one row per epoch (`currentImageName`, `currentBackgroundScale`, …) |
| `ds.cells` | DataFrame, one row per saved cell (cell_type, STA fit, EI stats) |
| `ds.spike_times(cell_id)` | list of arrays (ms), one per epoch |
| `ds.psth_matrix(cell_id)` | `(n_epochs, n_bins)` Hz, Gaussian-smoothed |
| `ds.psth_time_ms()` | shared bin-center time axis |

### Why HDF5 and not Parquet/Pickle

Ragged per-epoch spike-time arrays + a regular `(n_epochs, n_bins)`
PSTH matrix coexist naturally in HDF5 groups. Reload is ~0.2 s and
the file is portable across machines.

In [ ]:
# §10 — Build / load the offline store for one experiment.
# First call: 1–2 min; subsequent calls: <1 s.

EXP = '20221123C'
PROTOCOL = 'eye_movement_alt_bg'

ds = ra.load_or_build_offline(
    EXP, protocol=PROTOCOL,
    protocol_search='AlternatingBackground',
    overwrite=False, verbose=True,
)
print(ds)
print('cell types:', ds.cell_types())
display(ds.epochs.head())
display(ds.cells.head())


## 11. Offline analyses (`retinanalysis.protocols.eye_movement_alt_bg`)

Each analysis takes the `OfflineDataset` from §10 and returns a
DataFrame (or dict for population metrics). Results are saved next to
`offline.h5` so cross-date pooling in §12 is a single `pd.concat`.

| function | what it computes | output |
|---|---|---|
| `analyze_offline` | per-(cell-type × condition) mean PSTHs | dict (in-memory) |
| `spike_distance_analysis` | Victor-Purpura distance over a 5-s window per trial. Reports within-condition mean (variability inside a condition) and across-condition mean; `d_diff = d_across - d_within_avg > 0` means the condition modulates the response. | DataFrame + `spike_distance.csv` |
| `movie_repeat_analysis` | splits `stimTime` into cycle-1 vs cycle-2 (15 s + 15 s, drop first second). Reports per (cell, condition): correlation, RMSE, mean-rate ratio (adaptation index), and optional per-trial VP distance. | DataFrame + `movie_repeat.csv` |
| `population_time_scale_metrics` | time-resolved population-vector divergence between the two `currentBackgroundScale` levels per cell type (Cohen's d, Euclidean / cosine distance, cumulative |Δrate|, per-bin Mann-Whitney AUC). | dict (in-memory) |

These defaults are **tuned for `EyeMovementTrajectoryAlternatingBackground`**:
- Movie cycle = 15 s (one full Eye-Movement trajectory loop).
- Condition keys = `(currentImageName, currentBackgroundScale)`.
- Population comparison axis = `currentBackgroundScale`.

For a different protocol, write a new analyzer module under
`retinanalysis/protocols/<protocol_name>/` and call it from a
protocol-specific notebook.

In [ ]:
# §11a — Average PSTH by (cell type × condition). Offline = no DJ needed.
from retinanalysis.protocols import eye_movement_alt_bg as ema

r = ema.analyze_offline(ds, minimum_n=3)
print(f'cell types: {r["cell_types"]}')
print(f'{len(r["conditions"])} conditions, {len(r["time_ms"])} time bins')

ema.plot_psth_by_condition(r, show_individual_cells=False)


In [ ]:
# §11b — Movie-repeat comparison: cycle 1 vs cycle 2 (15s each, drop first 1s).
# compute_vp=False keeps it fast (~30 s); enable for per-trial VP timing differences.
mr = ema.movie_repeat_analysis(
    ds, cycle_sec=15.0, drop_first_sec=1.0,
    cell_types=['OnP', 'OffP', 'OnM', 'OffM'],
    compute_vp=False,
)
print(f'rows: {len(mr)}')
display(mr.groupby('cell_type')[['n_trials',
                                  'rate_cycle1_hz', 'rate_cycle2_hz',
                                  'rate_ratio',
                                  'corr_cycle12', 'rmse_cycle12_hz']]
          .median().round(3))

# Save to disk for cross-date pooling
ema.save_movie_repeat(mr, ds.exp_name)


In [ ]:
# §11c — Population time-scale metrics: two backgroundScale levels per cell type.
# Returns dict {cell_type: {cohens_d_mean, pop_euclid_dist, pop_cosine_dist, ...}}.

pm = ema.population_time_scale_metrics(
    ds, primary_key='currentBackgroundScale',
    cell_types=['OnP', 'OffP', 'OnM', 'OffM'],
    smooth_ms=100.0, minimum_n=3,
)
ema.plot_population_time_scale(pm, ds)


In [ ]:
# §11d — Spike-distance (Victor-Purpura) across backgroundScale, within image.
# Default pair_within=('currentImageName',): for each image, pair trials across
# the low vs high backgroundScale.  C-accelerated DP makes this fast — under 5s
# for ~170 cells × 5 images.
sd = ema.spike_distance_analysis(
    ds, window_sec=5.0, cost_per_sec=4.0,
    cell_types=['OnP', 'OffP', 'OnM', 'OffM'],
    pair_within=('currentImageName',),   # hold image constant, compare BG scale
    n_trials_cap=None,                    # use every trial — fast in C
)
print(f'rows: {len(sd)}; cells: {sd["cell_id"].nunique()}; '
      f'images: {sd["group_key"].nunique()}')

display(sd.groupby('cell_type')[['d_within_avg', 'd_across', 'd_diff']]
          .agg(['median', 'count']).round(2))

ema.save_spike_distance(sd, ds.exp_name)


## 12. Cross-date aggregation

Once every experiment has been through §10–§11 (each writes
`offline.h5`, `spike_distance.csv`, `movie_repeat.csv` to its own
folder), pooling across dates is just a `concat`.

| call | returns |
|---|---|
| `ra.load_offline_many()` | `{exp_name: OfflineDataset}` for every experiment with `offline.h5` on disk |
| `ema.aggregate_psth_across_dates(offlines)` | pooled per-cell mean PSTHs as one `(n_cells_total, n_bins)` matrix per `(cell_type, condition)` — pass to `ema.plot_psth_by_condition` |
| `ema.load_spike_distance_many()` | long-format DataFrame, `exp_name`-tagged |
| `ema.load_movie_repeat_many()` | long-format DataFrame, `exp_name`-tagged |

Adding a new date to the pool: run §10 (and §11 if you want the
spike-distance / movie-repeat CSVs) for that date, then re-run §12
unchanged — `load_*_many()` picks it up automatically.

In [ ]:
# §12 — Cross-date pooled analyses.
offlines = ra.load_offline_many()  # all dates with offline.h5
print(f'experiments loaded: {len(offlines)}')
for exp, ds_ in offlines.items():
    print(f'  {exp}: {len(ds_.cell_ids)} cells, types={ds_.cell_types()}')

# Pool PSTHs across dates
pooled = ema.aggregate_psth_across_dates(offlines, minimum_n=5)
print(f'\npooled types: {pooled["cell_types"]} from {pooled["n_dates"]} dates')
ema.plot_psth_by_condition(pooled, show_individual_cells=False)

# Pool spike-distance & movie-repeat CSVs
sd_all = ema.load_spike_distance_many()
mr_all = ema.load_movie_repeat_many()
print(f'\nspike_distance rows: {len(sd_all)} from '
      f'{sd_all["exp_name"].nunique() if not sd_all.empty else 0} dates')
print(f'movie_repeat rows: {len(mr_all)} from '
      f'{mr_all["exp_name"].nunique() if not mr_all.empty else 0} dates')

# Headline cross-date summaries
if not sd_all.empty:
    display(sd_all.groupby(['cell_type'])
                  [['d_within_avg', 'd_across', 'd_diff']]
                  .agg(['median', 'count']).round(2))
if not mr_all.empty:
    display(mr_all.groupby(['cell_type'])
                  [['rate_ratio', 'corr_cycle12', 'rmse_cycle12_hz']]
                  .agg(['median', 'count']).round(3))
